# 27 — Vectorized Portfolio Pricing

Batch-price entire portfolios using `jax.vmap` — no Python loops.

| Technique | Best For |
|-----------|----------|
| `vmap` | Same model, varying parameters |
| `pmap` | Multi-device (multi-GPU) |
| Scan / `lax.map` | Sequential dependencies |

In [ ]:
BACKEND = "cpu"

import sys, os, time
sys.path.insert(0, os.path.join(os.getcwd(), "..") if not os.getcwd().endswith("ql-jax") else ".")
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))

from notebooks._common import setup_backend, timed_ms, plot_speedup
jax = setup_backend(BACKEND)
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from ql_jax.engines.analytic.black_scholes_merton import bsm_price
from ql_jax.instruments.swap import swap_npv
from ql_jax.engines.analytic.heston import heston_price

---
## 1. vmap: 10,000 Options at Once

In [ ]:
N = 10_000
key = jax.random.PRNGKey(0)
keys = jax.random.split(key, 5)

S = jnp.full(N, 100.0)
K = jax.random.uniform(keys[0], (N,), minval=80.0, maxval=120.0)
T = jax.random.uniform(keys[1], (N,), minval=0.1, maxval=2.0)
r = jnp.full(N, 0.03)
q = jnp.full(N, 0.01)
sigma = jax.random.uniform(keys[2], (N,), minval=0.1, maxval=0.5)
cp = jnp.ones(N)  # all calls

# Vectorized pricing
batch_bsm = jax.vmap(bsm_price)
prices = batch_bsm(S, K, T, r, q, sigma, cp)

print(f"Portfolio size : {N:,}")
print(f"Total premium  : {float(jnp.sum(prices)):,.2f}")
print(f"Mean premium   : {float(jnp.mean(prices)):.4f}")

In [ ]:
# Python loop baseline
def loop_price(S, K, T, r, q, sigma, cp):
    return jnp.array([bsm_price(S[i], K[i], T[i], r[i], q[i], sigma[i], cp[i]) for i in range(len(S))])

# Force JIT compile
batch_bsm_jit = jax.jit(batch_bsm)
_ = batch_bsm_jit(S, K, T, r, q, sigma, cp)

ms_loop, _ = timed_ms(loop_price, S, K, T, r, q, sigma, cp)
ms_vmap, _ = timed_ms(batch_bsm_jit, S, K, T, r, q, sigma, cp)

print(f"Python loop : {ms_loop:.1f} ms")
print(f"vmap + JIT  : {ms_vmap:.1f} ms")
print(f"Speedup     : {ms_loop / ms_vmap:.0f}×")

---
## 2. Scaling: Speedup vs Portfolio Size

In [ ]:
sizes = [100, 500, 1000, 5000, 10_000, 50_000]
speedups = []

for n in sizes:
    S_n = jnp.full(n, 100.0)
    K_n = jnp.linspace(80, 120, n)
    T_n = jnp.full(n, 1.0)
    r_n = jnp.full(n, 0.03)
    q_n = jnp.full(n, 0.01)
    sig_n = jnp.full(n, 0.2)
    cp_n = jnp.ones(n)

    f_jit = jax.jit(jax.vmap(bsm_price))
    _ = f_jit(S_n, K_n, T_n, r_n, q_n, sig_n, cp_n)  # warm

    ms_v, _ = timed_ms(f_jit, S_n, K_n, T_n, r_n, q_n, sig_n, cp_n)
    # Estimate loop time for small sizes, extrapolate for large
    if n <= 1000:
        ms_l, _ = timed_ms(loop_price, S_n, K_n, T_n, r_n, q_n, sig_n, cp_n)
    else:
        ms_l = ms_l * n / 1000  # linear extrapolation
    speedups.append(ms_l / ms_v)

plt.figure(figsize=(8, 5))
plt.plot(sizes, speedups, 'o-', markersize=8)
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Portfolio Size')
plt.ylabel('Speedup (vmap+JIT / loop)')
plt.title('Vectorization Speedup vs Portfolio Size')
plt.grid(True, alpha=0.3)
plt.show()

---
## 3. Portfolio Greeks via AD + vmap

In [ ]:
# Price + all first-order Greeks for every trade in one pass
def price_and_greeks(S, K, T, r, q, sigma, cp):
    primals, tangents = jax.value_and_grad(bsm_price, argnums=(0, 5))(S, K, T, r, q, sigma, cp)
    delta = jax.grad(bsm_price, argnums=0)(S, K, T, r, q, sigma, cp)
    vega = jax.grad(bsm_price, argnums=5)(S, K, T, r, q, sigma, cp)
    theta = jax.grad(lambda t: bsm_price(S, K, t, r, q, sigma, cp))(T)
    rho = jax.grad(lambda rate: bsm_price(S, K, T, rate, q, sigma, cp))(r)
    return primals, delta, vega, theta, rho

batch_greeks = jax.jit(jax.vmap(price_and_greeks))

N_demo = 5000
S_d = jnp.full(N_demo, 100.0)
K_d = jnp.linspace(80, 120, N_demo)
T_d = jnp.full(N_demo, 1.0)
r_d = jnp.full(N_demo, 0.03)
q_d = jnp.full(N_demo, 0.01)
sig_d = jnp.full(N_demo, 0.2)
cp_d = jnp.ones(N_demo)

prices_d, deltas, vegas, thetas, rhos = batch_greeks(S_d, K_d, T_d, r_d, q_d, sig_d, cp_d)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
k_np = np.array(K_d)
for ax, vals, name in zip(axes.flat,
    [deltas, vegas, thetas, rhos],
    ['Delta', 'Vega', 'Theta', 'Rho']):
    ax.plot(k_np, np.array(vals))
    ax.set_xlabel('Strike')
    ax.set_title(name)
    ax.grid(True, alpha=0.3)
fig.suptitle('Portfolio Greeks (5,000 options)', fontsize=14)
fig.tight_layout()
plt.show()

---
## 4. Portfolio Risk Aggregation

In [ ]:
# Portfolio-level Greeks = sum of individual Greeks
portfolio_delta = float(jnp.sum(deltas))
portfolio_vega = float(jnp.sum(vegas))
portfolio_theta = float(jnp.sum(thetas))
portfolio_rho = float(jnp.sum(rhos))

print(f"Portfolio Delta : {portfolio_delta:+,.2f}")
print(f"Portfolio Vega  : {portfolio_vega:+,.2f}")
print(f"Portfolio Theta : {portfolio_theta:+,.2f}")
print(f"Portfolio Rho   : {portfolio_rho:+,.2f}")
print(f"Total Market Value: {float(jnp.sum(prices_d)):,.2f}")

---
## 5. Heston Portfolio — Advanced vmap

In [ ]:
N_h = 1000
K_h = jnp.linspace(80, 120, N_h)
T_h = jnp.full(N_h, 1.0)

heston_batch = jax.jit(jax.vmap(
    lambda K, T: heston_price(100.0, K, T, 0.03, 0.01, 0.04, 1.5, 0.04, 0.3, -0.7, 1)
))

_ = heston_batch(K_h, T_h)  # warm
ms_heston, prices_h = timed_ms(heston_batch, K_h, T_h)

print(f"1,000 Heston prices in {ms_heston:.1f} ms")

plt.figure(figsize=(8, 5))
plt.plot(np.array(K_h), np.array(prices_h))
plt.xlabel('Strike')
plt.ylabel('Call Price')
plt.title('Heston Portfolio: Vectorized Strike Ladder')
plt.grid(True, alpha=0.3)
plt.show()

---
## 6. Exercises

1. **Mixed portfolio**: Price a portfolio with both calls and puts using a single `vmap` call.
2. **Swap book**: Batch-price 1,000 IRS with varying fixed rates using `vmap`.
3. **Parallel Greeks**: Use `jax.jacfwd` to get all Greeks in a single forward pass per trade.

---
*End of Notebook 27*